# Grupo 12 — Servidor Flask (Estático + Dinâmico)

Este notebook substitui o `Grupo12_4.ipynb` original, adicionando suporte completo para **gestos dinâmicos** (LSTM + WLASL-100) além dos **gestos estáticos** (MLP + ASL Alphabet).

### Pré-requisitos
- `models/mlp_asl_landmarks.joblib` — do Grupo12_2
- `models/scaler_asl_landmarks.joblib` — do Grupo12_2
- `models/label_encoder_asl_landmarks.joblib` — do Grupo12_2
- `models/lstm_wlasl100_best.keras` — do Grupo12_4D
- `models/label_encoder_wlasl100.joblib` — do Grupo12_4D
- `models/lstm_norm_mean.npy` — do Grupo12_4D
- `models/lstm_norm_std.npy` — do Grupo12_4D

### Rotas da API
| Método | Rota | Descrição |
|--------|------|-----------|
| POST | `/api/predict` | Gestos estáticos (frame única → letra) |
| POST | `/api/predict_dynamic` | Gestos dinâmicos (sequência de frames → palavra) |
| GET  | `/api/health` | Estado do servidor |
| POST | `/api/llm_correct` | Correção de frase |
| POST | `/api/speak` | Síntese de voz |


In [1]:
# import subprocess, sys

# packages = [
#     'flask', 'flask-cors', 'pyttsx3', 'requests',
#     'numpy', 'opencv-python', 'mediapipe',
#     'joblib', 'textblob', 'tensorflow'
# ]

# for pkg in packages:
#     subprocess.check_call(
#         [sys.executable, '-m', 'pip', 'install', pkg, '-q', '--break-system-packages'],
#         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
#     )

# print('✅ Dependências verificadas.')


In [2]:
import os
import re
import base64
import threading
import json
import collections

import cv2
import numpy as np
import mediapipe as mp
import joblib
import requests
import pyttsx3

import tensorflow as tf
from tensorflow import keras

from textblob import TextBlob
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS

print('✅ Imports concluídos.')
print(f'   TensorFlow: {tf.__version__}')
print(f'   MediaPipe:  {mp.__version__}')


2026-04-07 16:19:04.355770: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-07 16:19:04.356457: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-07 16:19:04.359163: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-07 16:19:04.366027: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775575144.377909   49251 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775575144.38

✅ Imports concluídos.
   TensorFlow: 2.19.0
   MediaPipe:  0.10.21


In [3]:
# ════════════════════════════════════════════════════════════════════
# CAMINHOS DOS MODELOS
# ════════════════════════════════════════════════════════════════════

MODELS_DIR = 'models'

# ── Estático (MLP) ──────────────────────────────────────────────────
MLP_PATH    = os.path.join(MODELS_DIR, 'mlp_asl_landmarks.joblib')
SCALER_PATH = os.path.join(MODELS_DIR, 'scaler_asl_landmarks.joblib')
LE_PATH     = os.path.join(MODELS_DIR, 'label_encoder_asl_landmarks.joblib')

# ── Dinâmico (LSTM) ─────────────────────────────────────────────────
LSTM_PATH      = os.path.join(MODELS_DIR, 'lstm_wlasl100_best.keras')
LE_DYN_PATH    = os.path.join(MODELS_DIR, 'label_encoder_wlasl100.joblib')
NORM_MEAN_PATH = os.path.join(MODELS_DIR, 'lstm_norm_mean.npy')
NORM_STD_PATH  = os.path.join(MODELS_DIR, 'lstm_norm_std.npy')

# ── Verificar estáticos (obrigatórios) ──────────────────────────────
for p in [MLP_PATH, SCALER_PATH, LE_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f'Modelo estático não encontrado: {p}\nCorre o Grupo12_2.ipynb primeiro.'
        )

mlp           = joblib.load(MLP_PATH)
scaler        = joblib.load(SCALER_PATH)
label_encoder = joblib.load(LE_PATH)
print(f'✅ Modelo estático (MLP) carregado — {len(label_encoder.classes_)} classes')

# ── Verificar dinâmicos (opcionais — avisa mas não falha) ───────────
DYNAMIC_AVAILABLE = all(
    os.path.exists(p)
    for p in [LSTM_PATH, LE_DYN_PATH, NORM_MEAN_PATH, NORM_STD_PATH]
)

if DYNAMIC_AVAILABLE:
    try:
        lstm_model     = keras.models.load_model(LSTM_PATH, compile=False, safe_mode=False)
        le_dynamic     = joblib.load(LE_DYN_PATH)
        lstm_norm_mean = np.load(NORM_MEAN_PATH)
        lstm_norm_std  = np.load(NORM_STD_PATH)
        print(f'✅ Modelo dinâmico (LSTM) carregado — {len(le_dynamic.classes_)} gestos')
        print(f'   Gestos: {list(le_dynamic.classes_[:10])} ...')
    except Exception as e:
        print(f'⚠️ Erro ao carregar modelo dinâmico: {e}')
        print('⚠️ Vou continuar só com modo estático.')
        DYNAMIC_AVAILABLE = False
        lstm_model = le_dynamic = lstm_norm_mean = lstm_norm_std = None
else:
    lstm_model = le_dynamic = lstm_norm_mean = lstm_norm_std = None
    print('⚠️  Modelos dinâmicos não encontrados — modo estático apenas.')
    print('   Corre o Grupo12_3D + Grupo12_4D para ativar gestos dinâmicos.')


✅ Modelo estático (MLP) carregado — 26 classes


/home/tomas/MIA/1ANO/2SEM/SA/SA-25_26/TrabalhoPratico/aslai_server311/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelBinarizer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/tomas/MIA/1ANO/2SEM/SA/SA-25_26/TrabalhoPratico/aslai_server311/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MLPClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/tomas/MIA/1ANO/2SEM/SA/SA-25_26/TrabalhoPratico/aslai_server311/lib/python3.11/site-package

✅ Modelo dinâmico (LSTM) carregado — 100 gestos
   Gestos: ['accident', 'africa', 'all', 'apple', 'basketball', 'bed', 'before', 'bird', 'birthday', 'black'] ...


2026-04-07 16:19:06.895448: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
/home/tomas/MIA/1ANO/2SEM/SA/SA-25_26/TrabalhoPratico/aslai_server311/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
# ════════════════════════════════════════════════════════════════════
# MEDIAPIPE — Hands (estático) + Holistic (dinâmico)
# ════════════════════════════════════════════════════════════════════

mp_hands    = mp.solutions.hands
mp_drawing  = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

# ── Detetor de mãos para gestos ESTÁTICOS ───────────────────────────
# static_image_mode=True porque recebemos frames isoladas (não vídeo contínuo)
hands_detector = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ── Detetor Holistic para gestos DINÂMICOS ───────────────────────────
# static_image_mode=True pela mesma razão: frames individuais vindas do frontend
holistic_detector = mp_holistic.Holistic(
    static_image_mode=True,
    model_complexity=1,
    enable_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ── Constantes para a LSTM ───────────────────────────────────────────
SEQ_LEN    = 30
N_POSE     = 33 * 4   # 132
N_HAND     = 21 * 3   # 63
N_FEATURES = N_POSE + 2 * N_HAND  # 258

print('✅ MediaPipe iniciado (Hands + Holistic)')
print(f'   hands_detector   OK: {hands_detector is not None}')
print(f'   holistic_detector OK: {holistic_detector is not None}')


✅ MediaPipe iniciado (Hands + Holistic)
   hands_detector   OK: True
   holistic_detector OK: True


I0000 00:00:1775575147.078489   49251 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775575147.080492   49673 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
I0000 00:00:1775575147.096406   49251 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775575147.097457   49692 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)


W0000 00:00:1775575147.102571   49643 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [5]:
# ════════════════════════════════════════════════════════════════════
# NORMALIZAÇÃO DE LANDMARKS — estáticos (idêntica ao Grupo12_2)
# ════════════════════════════════════════════════════════════════════

def normalize_landmarks_static(lms):
    """
    Normalização idêntica à do Grupo12_2:
    63 coordenadas relativas ao pulso + 21 distâncias = 84 features.
    """
    wx, wy, wz = lms[0].x, lms[0].y, lms[0].z
    base_ids   = [5, 9, 13, 17]

    dists_base = [
        ((lms[i].x - wx)**2 + (lms[i].y - wy)**2 + (lms[i].z - wz)**2)**0.5
        for i in base_ids
    ]
    scale = float(np.mean(dists_base))
    if scale < 1e-6:
        return None

    features = []
    for lm in lms:
        features.extend([
            (lm.x - wx) / scale,
            (lm.y - wy) / scale,
            (lm.z - wz) / scale
        ])
    for lm in lms:
        d = ((lm.x - wx)**2 + (lm.y - wy)**2 + (lm.z - wz)**2)**0.5
        features.append(d / scale)

    return features  # 84 valores

print('✅ normalize_landmarks_static definida')


✅ normalize_landmarks_static definida


In [6]:
# ════════════════════════════════════════════════════════════════════
# PIPELINE ESTÁTICA — predict_from_frame
# ════════════════════════════════════════════════════════════════════

def predict_from_frame(frame_bgr):
    """
    Recebe um frame BGR, devolve dict:
      letter, confidence, hand_detected, annotated_frame_b64
    """
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    results   = hands_detector.process(frame_rgb)

    letter     = None
    confidence = 0.0
    hand_det   = False

    if results.multi_hand_landmarks:
        hand_det = True
        for hand_lm in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame_bgr, hand_lm, mp_hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 120), thickness=2, circle_radius=3),
                mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2)
            )
            features = normalize_landmarks_static(hand_lm.landmark)
            if features is not None:
                arr   = np.array(features).reshape(1, -1)
                arr_s = scaler.transform(arr)
                pred  = mlp.predict(arr_s)
                proba = mlp.predict_proba(arr_s).max()
                letter     = label_encoder.inverse_transform(pred)[0]
                confidence = float(proba)

    _, buf = cv2.imencode('.jpg', frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, 80])
    frame_b64 = base64.b64encode(buf).decode('utf-8')

    return {
        'letter': letter,
        'confidence': confidence,
        'hand_detected': hand_det,
        'annotated_frame_b64': frame_b64
    }

print('✅ Pipeline estática pronta')


✅ Pipeline estática pronta


W0000 00:00:1775575147.119699   49656 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [7]:
# ════════════════════════════════════════════════════════════════════
# PIPELINE DINÂMICA — buffer de frames + inferência LSTM
# ════════════════════════════════════════════════════════════════════

def extract_holistic_vector(frame_bgr):
    """
    Extrai vector de 258 features (pose + mãos) de uma frame BGR.
    Retorna também o frame anotado com os landmarks desenhados.
    """
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frame_rgb.flags.writeable = False
    results = holistic_detector.process(frame_rgb)
    frame_rgb.flags.writeable = True

    # ── Pose ──
    if results.pose_landmarks:
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility]
             for lm in results.pose_landmarks.landmark],
            dtype=np.float32
        ).flatten()
        mp_drawing.draw_landmarks(
            frame_bgr, results.pose_landmarks,
            mp_holistic.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(80, 110, 10), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(80, 256, 121), thickness=1)
        )
    else:
        pose = np.zeros(N_POSE, dtype=np.float32)

    # ── Mão esquerda ──
    if results.left_hand_landmarks:
        lh = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()
        mp_drawing.draw_landmarks(
            frame_bgr, results.left_hand_landmarks,
            mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2)
        )
    else:
        lh = np.zeros(N_HAND, dtype=np.float32)

    # ── Mão direita ──
    if results.right_hand_landmarks:
        rh = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()
        mp_drawing.draw_landmarks(
            frame_bgr, results.right_hand_landmarks,
            mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2)
        )
    else:
        rh = np.zeros(N_HAND, dtype=np.float32)

    vec = np.concatenate([pose, lh, rh])  # (258,)
    body_detected = (
        results.pose_landmarks is not None or
        results.left_hand_landmarks is not None or
        results.right_hand_landmarks is not None
    )

    return vec, body_detected, frame_bgr


def classify_sequence(sequence_list):
    """
    Recebe lista de vectores (N, 258), normaliza e classifica com a LSTM.
    Devolve (word, confidence, top5).
    """
    if not DYNAMIC_AVAILABLE:
        return None, 0.0, []

    arr = np.array(sequence_list, dtype=np.float32)  # (T, 258)
    T   = arr.shape[0]

    # Padding / truncamento a SEQ_LEN
    if T >= SEQ_LEN:
        start = (T - SEQ_LEN) // 2
        arr   = arr[start:start + SEQ_LEN]
    else:
        pad = np.tile(arr[-1:], (SEQ_LEN - T, 1))
        arr = np.vstack([arr, pad])

    # Normalização Z-score (parâmetros do treino)
    arr_n = (arr - lstm_norm_mean[0]) / lstm_norm_std[0]
    inp   = arr_n[np.newaxis, ...]  # (1, SEQ_LEN, 258)

    probs    = lstm_model.predict(inp, verbose=0)[0]  # (N_CLASSES,)
    top5_idx = probs.argsort()[-5:][::-1]

    predicted_idx  = top5_idx[0]
    predicted_word = le_dynamic.classes_[predicted_idx]
    confidence     = float(probs[predicted_idx])
    top5 = [
        {'word': le_dynamic.classes_[i], 'confidence': float(probs[i])}
        for i in top5_idx
    ]

    return predicted_word, confidence, top5

print('✅ Pipeline dinâmica pronta')


✅ Pipeline dinâmica pronta


In [8]:
# ════════════════════════════════════════════════════════════════════
# LLM + CORREÇÃO DE FRASE (mantido do Grupo12_4 original)
# ════════════════════════════════════════════════════════════════════

OLLAMA_URL = 'http://localhost:11434/api/generate'

PREFERRED_OLLAMA_MODELS = [
    'qwen2.5:3b-instruct', 'llama3.2:3b', 'phi3:mini', 'llama3'
]

COMMON_REWRITE_RULES = {
    'i want water':      'I want some water, please.',
    'i want some water': 'I want some water, please.',
    'i want eat':        'I would like something to eat.',
    'i want to eat':     'I would like something to eat.',
    'want eat':          'I would like something to eat.',
    'love you':          'I love you.',
}


def check_ollama_available():
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        models = [m['name'] for m in r.json().get('models', [])]
        return True, models
    except Exception:
        return False, []


def select_ollama_model(models):
    if not models:
        return None
    for pref in PREFERRED_OLLAMA_MODELS:
        for m in models:
            if m == pref or m.startswith(pref):
                return m
    return models[0]


def clean_phrase_spacing(text):
    return re.sub(r'\s+', ' ', (text or '').strip())


def basic_spell_correct_phrase(raw_phrase):
    cleaned   = clean_phrase_spacing(raw_phrase)
    if not cleaned:
        return ''
    corrected = str(TextBlob(cleaned.lower()).correct())
    corrected = clean_phrase_spacing(corrected)
    corrected = re.sub(r'\bi\b', 'I', corrected)
    if corrected:
        corrected = corrected[0].upper() + corrected[1:]
    return corrected


def apply_common_rewrite_rules(text):
    key = re.sub(r'[^a-z ]+', '', clean_phrase_spacing(text).lower())
    return COMMON_REWRITE_RULES.get(clean_phrase_spacing(key))


def sanitize_llm_text(text):
    if not text:
        return ''
    cleaned = clean_phrase_spacing(text).strip('\"\' ')
    lines   = [ln.strip('\"\' ') for ln in cleaned.splitlines() if ln.strip()]
    if lines:
        cleaned = lines[0]
    cleaned = re.sub(r'\bi\b', 'I', clean_phrase_spacing(cleaned))
    if cleaned:
        cleaned = cleaned[0].upper() + cleaned[1:]
    return cleaned


def llm_correct_phrase(raw_phrase):
    pre_corrected = basic_spell_correct_phrase(raw_phrase)
    rule_result   = apply_common_rewrite_rules(pre_corrected)
    if rule_result:
        return {'corrected': rule_result, 'strategy': 'rule', 'pre_corrected': pre_corrected}

    ollama_ok, ollama_models = check_ollama_available()
    if ollama_ok:
        selected = select_ollama_model(ollama_models)
        prompt = (
            'You are an ASL post-processor. Fix spelling and grammar of the following '
            'sentence. Return ONLY the corrected sentence, nothing else.\n\n'
            f'Input: {pre_corrected}\nOutput:'
        )
        try:
            r = requests.post(OLLAMA_URL, json={
                'model': selected, 'prompt': prompt, 'stream': False
            }, timeout=10)
            raw = r.json().get('response', '')
            corrected = sanitize_llm_text(raw)
            if corrected:
                return {'corrected': corrected, 'strategy': 'llm',
                        'model_used': selected, 'pre_corrected': pre_corrected}
        except Exception:
            pass

    return {'corrected': pre_corrected, 'strategy': 'spell_correct',
            'pre_corrected': pre_corrected}


print('✅ Módulo de correção de frase pronto')


✅ Módulo de correção de frase pronto


In [9]:
# ════════════════════════════════════════════════════════════════════
# TTS — pyttsx3 (mantido do original)
# ════════════════════════════════════════════════════════════════════

_tts_lock = threading.Lock()

def choose_english_voice(engine):
    voices    = engine.getProperty('voices')
    preferred = ['en-gb', 'english_rp', 'english-us', 'en-us', 'english', 'gmw/en']
    for pref in preferred:
        for v in voices:
            if pref in f"{getattr(v, 'id', '')} {getattr(v, 'name', '')}".lower():
                return v.id
    for v in voices:
        if 'en' in f"{getattr(v, 'id', '')} {getattr(v, 'name', '')}".lower():
            return v.id
    return None


def speak_text(text, rate=150, volume=1.0):
    try:
        with _tts_lock:
            engine = pyttsx3.init()
            engine.setProperty('rate', rate)
            engine.setProperty('volume', volume)
            vid = choose_english_voice(engine)
            if vid:
                engine.setProperty('voice', vid)
            engine.say(text)
            engine.runAndWait()
            engine.stop()
        return {'success': True, 'text_spoken': text}
    except Exception as e:
        return {'success': False, 'error': str(e)}


def speak_async(text):
    threading.Thread(target=speak_text, args=(text,), daemon=True).start()


print('✅ TTS (pyttsx3) pronto')


✅ TTS (pyttsx3) pronto


W0000 00:00:1775575147.162055   49676 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [10]:
# ════════════════════════════════════════════════════════════════════
# FLASK APP
# ════════════════════════════════════════════════════════════════════

os.makedirs('web', exist_ok=True)
app = Flask(__name__, static_folder='web')
CORS(app)


# ── Health check ──────────────────────────────────────────────────────
@app.route('/api/health', methods=['GET'])
def health():
    ollama_ok, ollama_models = check_ollama_available()
    return jsonify({
        'status': 'ok',
        'static_model': {
            'loaded': True,
            'type': 'MLP',
            'classes': list(label_encoder.classes_)
        },
        'dynamic_model': {
            'loaded': DYNAMIC_AVAILABLE,
            'type': 'LSTM',
            'classes': list(le_dynamic.classes_) if DYNAMIC_AVAILABLE else []
        },
        'ollama': {
            'available': ollama_ok,
            'models': ollama_models,
            'selected_model': select_ollama_model(ollama_models) if ollama_ok else None
        },
        'tts': 'pyttsx3'
    })


# ── Predição ESTÁTICA (frame única → letra) ──────────────────────────
@app.route('/api/predict', methods=['POST'])
def predict_static():
    """
    Body: { "frame": "<base64 JPEG>" }
    Resposta: { "letter": "A", "confidence": 0.97, "hand_detected": true,
                "annotated_frame": "<base64 JPEG>" }
    """
    data      = request.get_json(force=True)
    frame_b64 = data.get('frame', '')
    if not frame_b64:
        return jsonify({'error': 'Campo frame em falta.'}), 400

    try:
        img_bytes = base64.b64decode(frame_b64)
        img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
        frame_bgr = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        frame_bgr = cv2.flip(frame_bgr, 1)
    except Exception as e:
        return jsonify({'error': f'Frame inválido: {e}'}), 400

    try:
        result = predict_from_frame(frame_bgr)
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

    return jsonify({
        'letter':          result['letter'],
        'confidence':      result['confidence'],
        'hand_detected':   result['hand_detected'],
        'annotated_frame': result['annotated_frame_b64']
    })


# ── Predição DINÂMICA (sequência de frames → palavra) ─────────────────
@app.route('/api/predict_dynamic', methods=['POST'])
def predict_dynamic():
    """
    Body: { "frames": ["<base64 JPEG>", ...] }

    Resposta:
    {
      "word": "accident",
      "confidence": 0.72,
      "top5": [{"word": "accident", "confidence": 0.72}, ...],
      "body_detected": true,
      "annotated_frame": "<base64 JPEG última frame anotada>",
      "dynamic_available": true
    }
    """
    if not DYNAMIC_AVAILABLE:
        return jsonify({
            'word': None, 'confidence': 0.0, 'top5': [],
            'body_detected': False, 'dynamic_available': False,
            'error': 'Modelo dinâmico não carregado. Corre Grupo12_4D primeiro.'
        }), 503

    data       = request.get_json(force=True)
    frames_b64 = data.get('frames', [])

    if len(frames_b64) < 5:
        return jsonify({'error': 'Mínimo 5 frames necessárias.'}), 400

    sequence       = []
    last_annotated = None
    body_detected  = False

    for fb64 in frames_b64:
        try:
            img_bytes = base64.b64decode(fb64)
            img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
            frame_bgr = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
            frame_bgr = cv2.flip(frame_bgr, 1)
        except Exception:
            continue

        vec, detected, annotated_frame = extract_holistic_vector(frame_bgr)
        sequence.append(vec)
        if detected:
            body_detected  = True
            last_annotated = annotated_frame

    if not sequence:
        return jsonify({'error': 'Nenhuma frame válida processada.'}), 400

    word, confidence, top5 = classify_sequence(sequence)

    if last_annotated is not None:
        _, buf = cv2.imencode('.jpg', last_annotated, [cv2.IMWRITE_JPEG_QUALITY, 75])
        annotated_b64 = base64.b64encode(buf).decode('utf-8')
    else:
        annotated_b64 = ''

    return jsonify({
        'word':              word,
        'confidence':        confidence,
        'top5':              top5,
        'body_detected':     body_detected,
        'annotated_frame':   annotated_b64,
        'dynamic_available': True
    })


# ── Correção de frase ─────────────────────────────────────────────────
@app.route('/api/llm_correct', methods=['POST'])
def llm_correct_route():
    data   = request.get_json(force=True)
    phrase = data.get('phrase', '').strip()
    if not phrase:
        return jsonify({'error': 'Frase em falta.'}), 400
    return jsonify(llm_correct_phrase(phrase))


# ── TTS ───────────────────────────────────────────────────────────────
@app.route('/api/speak', methods=['POST'])
def speak_route():
    data = request.get_json(force=True)
    text = data.get('text', '').strip()
    rate = int(data.get('rate', 150))
    if not text:
        return jsonify({'error': 'Texto em falta.'}), 400
    speak_async(text)
    return jsonify({'success': True, 'text_spoken': text})


# ── Servir frontend ───────────────────────────────────────────────────
@app.route('/', defaults={'path': ''})
@app.route('/<path:path>')
def serve_web(path):
    if path and os.path.exists(os.path.join('web', path)):
        return send_from_directory('web', path)
    return send_from_directory('web', 'index.html')


print('✅ Rotas Flask definidas:')
print('   GET  /api/health')
print('   POST /api/predict          ← estático (MLP)')
print('   POST /api/predict_dynamic  ← dinâmico (LSTM)  *** NOVO ***')
print('   POST /api/llm_correct')
print('   POST /api/speak')
print('   GET  / → serve web/index.html')


✅ Rotas Flask definidas:
   GET  /api/health
   POST /api/predict          ← estático (MLP)
   POST /api/predict_dynamic  ← dinâmico (LSTM)  *** NOVO ***
   POST /api/llm_correct
   POST /api/speak
   GET  / → serve web/index.html


In [11]:
# ── Iniciar servidor ──────────────────────────────────────────────────
PORT = 5000
print(f'🚀 Servidor em http://localhost:{PORT}')
print(f'   Interface:  http://localhost:{PORT}/')
print(f'   Health:     http://localhost:{PORT}/api/health')
print()
print('   ⚠️  Para parar: Kernel → Interrupt (botão ■)')

app.run(
    host='0.0.0.0',
    port=PORT,
    debug=False,
    use_reloader=False,
    threaded=True
)


🚀 Servidor em http://localhost:5000
   Interface:  http://localhost:5000/
   Health:     http://localhost:5000/api/health

   ⚠️  Para parar: Kernel → Interrupt (botão ■)
 * Serving Flask app '__main__'
 * Debug mode: off


W0000 00:00:1775575147.193313   49680 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775575147.195506   49683 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775575147.195534   49688 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775575147.195541   49682 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.87:5000
Press CTRL+C to quit
W0000 00:00:1775575147.203745   49674 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inferen